In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

DATASET_DIR = PROJECT_DIR / "data" / "sample_dataset"
CHROMA_DIR = PROJECT_DIR / "data" / "chroma_db"
SRC_DIR = PROJECT_DIR / "src"

print("Project root:", PROJECT_DIR)
print("Dataset exists:", DATASET_DIR.exists())
print("ChromaDB exists:", CHROMA_DIR.exists())
print("Source directory exists:", SRC_DIR.exists())

Project root: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project
Dataset exists: True
ChromaDB exists: True
Source directory exists: True


In [2]:
from datasets import load_from_disk

dataset = load_from_disk(str(DATASET_DIR))

print("Dataset splits:", list(dataset.keys()))

for split_name, split in dataset.items():
    print(f"\n{split_name.upper()}")
    print("Number of patents:", f"{len(split):,}")
    print("Columns:", split.column_names)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Dataset splits: ['train', 'validation']

TRAIN
Number of patents: 16,153
Columns: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id']

VALIDATION
Number of patents: 9,094
Columns: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id']


In [3]:
sample = dataset["train"][0]

for field in ["patent_number", "title", "abstract", "claims", "description"]:
    value = sample.get(field)
    text = str(value or "")

    print(f"\n{'=' * 60}")
    print(f"{field.upper()} — {len(text):,} characters")
    print(text[:500] if text else "[EMPTY]")


PATENT_NUMBER — 8 characters
13261748

TITLE — 35 characters
MINI-OPTICAL NETWORK TERMINAL (ONT)

ABSTRACT — 544 characters
The present invention relates to passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. In one embodiment, the optical network terminal includes a first interface coupled to a communications network, a second interface coupled to a network client and a processor including a memory coupled to the first interface and to the second interface, wherein the processor is capable of converting optical signals to electric signals, such that the network c

CLAIMS — 2,494 characters
1. A compact optical network terminal, comprising: a first interface coupled to a communications network; a second interface coupled to a network client, wherein the second interface is a network connectivity dongle with an optical transceiver at one end; and a processor including a circuitry and a memory coupled to the first interface and to th

In [4]:
import chromadb

CHROMA_DIR = PROJECT_DIR / "data" / "chroma_db"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

for collection in client.list_collections():
    print(f"{collection.name}: {collection.count():,} chunks")

patent_rag: 1,000 chunks
patent_rag_chunks: 67,303 chunks
patent_sample_2016_chunks: 698 chunks


In [5]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().parent

EXPERIMENT_DIR = PROJECT_DIR / "data" / "embedding_experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "baseline": {
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "embedding_dimension": 384,
        "collection_name": "patent_sample_2016_chunks",
    },

    "candidate_models": [
        "AI-Growth-Lab/PatentSBERTa",
        "JHU-CLSP/PatentSBERTa",
        "nomic-ai/nomic-embed-text-v1.5",
        "BAAI/bge-base-en-v1.5",
        "intfloat/e5-base-v2",
        "sentence-transformers/all-mpnet-base-v2",
    ],

    "dataset": {
        "path": str(PROJECT_DIR / "data" / "sample_dataset"),
        "text_fields": ["title", "abstract"],
        "train_split": "train",
        "validation_split": "validation",
    },

    "chunking": {
        "max_characters": 1500,
        "overlap_characters": 200,
    },

    "evaluation": {
        "top_k": [1, 5, 10],
        "query_count": 10,
        "metrics": [
            "precision_at_k",
            "recall_at_k",
            "ndcg_at_k",
            "query_latency",
        ],
    },

    "resources": {
        "device": "cpu",
        "models_loaded_simultaneously": 1,
        "batch_size": 2,
    },
}

config_path = EXPERIMENT_DIR / "experiment_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2))

print("Configuration saved to:", config_path)
print("Candidate models:", len(CONFIG["candidate_models"]))
print("Models loaded simultaneously:", 
      CONFIG["resources"]["models_loaded_simultaneously"])

Configuration saved to: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/embedding_experiments/experiment_config.json
Candidate models: 6
Models loaded simultaneously: 1


In [6]:
CONFIG["dataset"]["text_fields"] = [
    "title",
    "abstract",
    "description",
]

CONFIG["evaluation"]["query_count"] = 10

config_path = EXPERIMENT_DIR / "experiment_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2))

print("Text fields:", CONFIG["dataset"]["text_fields"])
print("Configuration saved:", config_path)

Text fields: ['title', 'abstract', 'description']
Configuration saved: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/embedding_experiments/experiment_config.json


In [7]:
print("Train columns:")
print(dataset["train"].column_names)

if "embedding_text" in dataset["train"].column_names:
    print("\nEmbedding text column exists.")

    sample_text = dataset["train"][0]["embedding_text"]
    print("Characters:", len(sample_text or ""))
    print("Preview:")
    print((sample_text or "")[:1000])
else:
    print("\nNo embedding_text column found in the loaded dataset.")

Train columns:
['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id']

No embedding_text column found in the loaded dataset.


In [8]:
from pathlib import Path

script_path = PROJECT_DIR / "src" / "prepare_and_index_hupd.py"
script = script_path.read_text()

for i, line in enumerate(script.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "embedding_text",
        "description",
        "abstract",
        "chunk",
        "text_fields",
    ]):
        print(f"{i}: {line}")

10:     .../python src/prepare_and_index_hupd.py --batch-size 16 --chunk-size 1800
13: The script is resumable: deterministic chunk IDs and collection.upsert() mean
14: re-running it updates the same chunks rather than duplicating them.
30: DEFAULT_COLLECTION = "patent_sample_2016_chunks"
35: TEXT_FIELDS = (
37:     "abstract",
41:     "description",
57:         description="Validate, chunk, embed, and index an on-disk HUPD DatasetDict."
64:     parser.add_argument("--chunk-size", type=int, default=1800)
65:     parser.add_argument("--chunk-overlap", type=int, default=250)
115:     for field in TEXT_FIELDS:
124: def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
125:     """Create overlapping character chunks, preferring a nearby word boundary."""
128:     if len(text) <= chunk_size:
131:     chunks: list[str] = []
135:         end = min(start + chunk_size, text_length)
138:             # quarter of the chunk; otherwise use the hard boundary.
139:             lower_

In [9]:
lines = script.splitlines()

for start, end in [(30, 45), (100, 125)]:
    print(f"\n{'=' * 20} Lines {start}–{end} {'=' * 20}")

    for line_number in range(start, end + 1):
        if line_number <= len(lines):
            print(f"{line_number}: {lines[line_number - 1]}")


==================== Lines 30–45 ====================
30: DEFAULT_COLLECTION = "patent_sample_2016_chunks"
31: DEFAULT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
32: 
33: # Prefer the richer text fields, while gracefully skipping fields absent in
34: # a particular version of the dataset.
35: TEXT_FIELDS = (
36:     "title",
37:     "abstract",
38:     "claims",
39:     "background",
40:     "summary",
41:     "description",
42: )
43: METADATA_FIELDS = (
44:     "patent_number",
45:     "decision",

==================== Lines 100–125 ====================
100:     """Chroma metadata must contain simple scalar values, not lists/dicts."""
101:     if value is None:
102:         return ""
103:     if isinstance(value, (str, int, float, bool)):
104:         try:
105:             if isinstance(value, float) and value != value:
106:                 return ""
107:         except Exception:
108:             pass
109:         return value
110:     return as_text(value)
111: 
112: 
113: de

In [10]:
TEXT_FIELDS = (
    "title",
    "abstract",
    "claims",
    "background",
    "summary",
    "description",
)

In [11]:
CONFIG["dataset"]["text_fields"] = [
    "title",
    "abstract",
    "claims",
    "background",
    "summary",
    "description",
]

CONFIG["chunking"]["max_characters"] = 1800
CONFIG["chunking"]["overlap_characters"] = 250

config_path = EXPERIMENT_DIR / "experiment_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2))

print("Text fields:", CONFIG["dataset"]["text_fields"])
print("Chunk size:", CONFIG["chunking"]["max_characters"])
print("Chunk overlap:", CONFIG["chunking"]["overlap_characters"])
print("Configuration saved:", config_path)

Text fields: ['title', 'abstract', 'claims', 'background', 'summary', 'description']
Chunk size: 1800
Chunk overlap: 250
Configuration saved: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/embedding_experiments/experiment_config.json


In [12]:
from transformers import AutoTokenizer

for model_name in [
    CONFIG["baseline"]["model_name"],
    *CONFIG["candidate_models"],
]:
    print(f"\nModel: {model_name}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=False,
        )

        print("Tokenizer limit:", tokenizer.model_max_length)
        print("Special tokens:", tokenizer.num_special_tokens_to_add())
    except Exception as e:
        print("Could not inspect tokenizer:", str(e)[:200])


Model: sentence-transformers/all-MiniLM-L6-v2


Tokenizer limit: 512
Special tokens: 2

Model: AI-Growth-Lab/PatentSBERTa
Tokenizer limit: 512
Special tokens: 2

Model: JHU-CLSP/PatentSBERTa
Could not inspect tokenizer: JHU-CLSP/PatentSBERTa is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission t

Model: nomic-ai/nomic-embed-text-v1.5
Tokenizer limit: 8192
Special tokens: 2

Model: BAAI/bge-base-en-v1.5
Tokenizer limit: 512
Special tokens: 2

Model: intfloat/e5-base-v2
Tokenizer limit: 512
Special tokens: 2

Model: sentence-transformers/all-mpnet-base-v2
Tokenizer limit: 512
Special tokens: 2


In [13]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def make_document(record):
    sections = []

    for field in CONFIG["dataset"]["text_fields"]:
        value = str(record.get(field) or "").strip()

        if value:
            sections.append(f"{field.upper()}:\n{value}")

    return "\n\n".join(sections).strip()

token_lengths = []

for i in range(min(20, len(dataset["train"]))):
    record = dataset["train"][i]
    document = make_document(record)

    chunks = script  # Existing script source, for reference only
    # Use the same chunking function as the indexing script.
    # We'll extract and execute nothing from the script dynamically here.
    from importlib.util import spec_from_file_location, module_from_spec

    spec = spec_from_file_location("patent_indexer", PROJECT_DIR / "src" / "prepare_and_index_hupd.py")
    indexer = module_from_spec(spec)
    spec.loader.exec_module(indexer)

    patent_chunks = indexer.chunk_text(document, 1800, 250)

    for chunk in patent_chunks:
        count = len(tokenizer.encode(chunk, add_special_tokens=True))
        token_lengths.append(count)

print("Chunks measured:", len(token_lengths))
print("Minimum tokens:", min(token_lengths))
print("Median tokens:", int(np.median(token_lengths)))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Maximum tokens:", max(token_lengths))
print("Chunks exceeding 512 tokens:", sum(n > 512 for n in token_lengths))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (601 > 512). Running this sequence through the model will result in indexing errors


Chunks measured: 769
Minimum tokens: 83
Median tokens: 379
95th percentile: 460
Maximum tokens: 669
Chunks exceeding 512 tokens: 8


In [14]:
from sentence_transformers import SentenceTransformer
from transformers import AutoConfig

for model_name in CONFIG["candidate_models"]:
    print(f"\n{'=' * 60}")
    print(model_name)

    try:
        model_config = AutoConfig.from_pretrained(
            model_name,
            trust_remote_code=False,
        )

        print("Architecture:", model_config.model_type)
        print("Hidden size:", getattr(model_config, "hidden_size", "unknown"))
        print(
            "Configured max positions:",
            getattr(model_config, "max_position_embeddings", "unknown"),
        )

    except Exception as exc:
        print("Configuration unavailable:", str(exc)[:200])


AI-Growth-Lab/PatentSBERTa
Architecture: mpnet
Hidden size: 768
Configured max positions: 514

JHU-CLSP/PatentSBERTa
Configuration unavailable: JHU-CLSP/PatentSBERTa is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission t

nomic-ai/nomic-embed-text-v1.5
Architecture: nomic_bert
Hidden size: 768
Configured max positions: 2048

BAAI/bge-base-en-v1.5
Architecture: bert
Hidden size: 768
Configured max positions: 512

intfloat/e5-base-v2
Architecture: bert
Hidden size: 768
Configured max positions: 512

sentence-transformers/all-mpnet-base-v2
Architecture: mpnet
Hidden size: 768
Configured max positions: 514


In [15]:
MODEL_NOTES = {
    "sentence-transformers/all-MiniLM-L6-v2": {
        "role": "baseline",
        "hidden_size": 384,
        "configured_max_positions": 512,
    },
    "AI-Growth-Lab/PatentSBERTa": {
        "role": "patent-domain candidate",
        "hidden_size": 768,
        "configured_max_positions": 514,
    },
    "nomic-ai/nomic-embed-text-v1.5": {
        "role": "longer-context candidate",
        "hidden_size": 768,
        "configured_max_positions": 2048,
    },
    "BAAI/bge-base-en-v1.5": {
        "role": "general retrieval candidate",
        "hidden_size": 768,
        "configured_max_positions": 512,
    },
    "intfloat/e5-base-v2": {
        "role": "general retrieval candidate",
        "hidden_size": 768,
        "configured_max_positions": 512,
    },
    "sentence-transformers/all-mpnet-base-v2": {
        "role": "general semantic candidate",
        "hidden_size": 768,
        "configured_max_positions": 514,
    },
}

CONFIG["candidate_models"] = [
    "AI-Growth-Lab/PatentSBERTa",
    "nomic-ai/nomic-embed-text-v1.5",
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-base-v2",
    "sentence-transformers/all-mpnet-base-v2",
]

config_path.write_text(json.dumps(CONFIG, indent=2))

notes_path = EXPERIMENT_DIR / "model_notes.json"
notes_path.write_text(json.dumps(MODEL_NOTES, indent=2))

print("Candidate count:", len(CONFIG["candidate_models"]))
print("Model notes saved:", notes_path)

Candidate count: 5
Model notes saved: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/embedding_experiments/model_notes.json


In [16]:
from pathlib import Path

meminfo = {}

with open("/proc/meminfo") as f:
    for line in f:
        key, value = line.split(":", 1)
        meminfo[key] = int(value.strip().split()[0])  # kB

available_gib = meminfo["MemAvailable"] / (1024 ** 2)
total_gib = meminfo["MemTotal"] / (1024 ** 2)

print(f"Total RAM: {total_gib:.2f} GiB")
print(f"Available RAM: {available_gib:.2f} GiB")
print(f"RAM currently used: {total_gib - available_gib:.2f} GiB")

Total RAM: 7.56 GiB
Available RAM: 3.43 GiB
RAM currently used: 4.13 GiB


In [17]:
from pathlib import Path

cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
model_folder = cache_dir / "models--AI-Growth-Lab--PatentSBERTa"

if model_folder.exists():
    files = [
        p for p in model_folder.rglob("*")
        if p.is_file() and "snapshots" in p.parts
    ]

    print("Cached files:", len(files))

    for path in sorted(files, key=lambda p: p.stat().st_size, reverse=True)[:10]:
        size_mib = path.stat().st_size / (1024 ** 2)
        print(f"{size_mib:8.2f} MiB  {path.name}")
else:
    print("PatentSBERTa model weights are not fully cached yet.")
    print("Cache path:", model_folder)

Cached files: 5
    0.44 MiB  tokenizer.json
    0.22 MiB  vocab.txt
    0.00 MiB  config.json
    0.00 MiB  tokenizer_config.json
    0.00 MiB  special_tokens_map.json


In [18]:
from pathlib import Path

cache_dir = Path.home() / ".cache" / "huggingface" / "hub"

model_folders = {
    "PatentSBERTa": "models--AI-Growth-Lab--PatentSBERTa",
    "Nomic": "models--nomic-ai--nomic-embed-text-v1.5",
    "BGE": "models--BAAI--bge-base-en-v1.5",
    "E5": "models--intfloat--e5-base-v2",
    "MPNet": "models--sentence-transformers--all-mpnet-base-v2",
}

for name, folder_name in model_folders.items():
    folder = cache_dir / folder_name
    total_bytes = sum(
        p.stat().st_size
        for p in folder.rglob("*")
        if p.is_file() and "snapshots" in p.parts
    ) if folder.exists() else 0

    print(f"{name:12} {total_bytes / (1024**2):8.2f} MiB cached")

PatentSBERTa     0.67 MiB cached
Nomic            0.90 MiB cached
BGE              0.90 MiB cached
E5               0.90 MiB cached
MPNet            0.67 MiB cached


In [19]:
MODEL_INPUT_GUIDE = {
    "AI-Growth-Lab/PatentSBERTa": {
        "query_prefix": "",
        "document_prefix": "",
    },
    "nomic-ai/nomic-embed-text-v1.5": {
        "query_prefix": "search_query: ",
        "document_prefix": "search_document: ",
    },
    "BAAI/bge-base-en-v1.5": {
        "query_prefix": "",
        "document_prefix": "",
    },
    "intfloat/e5-base-v2": {
        "query_prefix": "query: ",
        "document_prefix": "passage: ",
    },
    "sentence-transformers/all-mpnet-base-v2": {
        "query_prefix": "",
        "document_prefix": "",
    },
}

guide_path = EXPERIMENT_DIR / "model_input_guide.json"
guide_path.write_text(json.dumps(MODEL_INPUT_GUIDE, indent=2))

print("Saved input-format guide:", guide_path)
for model, settings in MODEL_INPUT_GUIDE.items():
    print(f"{model}: {settings}")

Saved input-format guide: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/embedding_experiments/model_input_guide.json
AI-Growth-Lab/PatentSBERTa: {'query_prefix': '', 'document_prefix': ''}
nomic-ai/nomic-embed-text-v1.5: {'query_prefix': 'search_query: ', 'document_prefix': 'search_document: '}
BAAI/bge-base-en-v1.5: {'query_prefix': '', 'document_prefix': ''}
intfloat/e5-base-v2: {'query_prefix': 'query: ', 'document_prefix': 'passage: '}
sentence-transformers/all-mpnet-base-v2: {'query_prefix': '', 'document_prefix': ''}


In [20]:
TEST_QUERIES = [
    {
        "query": "Optical network terminals for passive optical networks",
        "topic": "optical networking",
    },
    {
        "query": "Methods for treating rosacea using topical compositions",
        "topic": "medical treatment",
    },
    {
        "query": "Machine learning systems for detecting fraudulent transactions",
        "topic": "machine learning",
    },
    {
        "query": "Battery thermal management systems for electric vehicles",
        "topic": "battery engineering",
    },
    {
        "query": "Wireless communication devices with improved antenna performance",
        "topic": "wireless communication",
    },
    {
        "query": "Methods for manufacturing semiconductor devices",
        "topic": "semiconductor manufacturing",
    },
    {
        "query": "Image processing techniques for medical diagnosis",
        "topic": "medical imaging",
    },
    {
        "query": "Systems for reducing energy consumption in data centers",
        "topic": "energy efficiency",
    },
    {
        "query": "Drug delivery systems using controlled-release formulations",
        "topic": "drug delivery",
    },
    {
        "query": "Methods for improving cybersecurity using anomaly detection",
        "topic": "cybersecurity",
    },
]

print("Number of test queries:", len(TEST_QUERIES))

for i, item in enumerate(TEST_QUERIES, start=1):
    print(f"{i:02}. [{item['topic']}] {item['query']}")

Number of test queries: 10
01. [optical networking] Optical network terminals for passive optical networks
02. [medical treatment] Methods for treating rosacea using topical compositions
03. [machine learning] Machine learning systems for detecting fraudulent transactions
04. [battery engineering] Battery thermal management systems for electric vehicles
05. [wireless communication] Wireless communication devices with improved antenna performance
06. [semiconductor manufacturing] Methods for manufacturing semiconductor devices
07. [medical imaging] Image processing techniques for medical diagnosis
08. [energy efficiency] Systems for reducing energy consumption in data centers
09. [drug delivery] Drug delivery systems using controlled-release formulations
10. [cybersecurity] Methods for improving cybersecurity using anomaly detection


In [21]:
import re
from collections import Counter

def tokenize(text):
    return set(re.findall(r"\b[a-z0-9]+\b", text.lower()))

# Search the validation split first to keep evaluation separate.
validation = dataset[CONFIG["dataset"]["validation_split"]]

search_fields = ["title", "abstract", "claims", "description"]

def lexical_candidates(query, limit=5):
    query_terms = tokenize(query)
    results = []

    for i, patent in enumerate(validation):
        combined = " ".join(
            str(patent.get(field) or "")
            for field in search_fields
        )
        patent_terms = tokenize(combined)
        overlap = len(query_terms & patent_terms)

        if overlap:
            results.append({
                "row": i,
                "patent_number": patent["patent_number"],
                "title": patent["title"],
                "cpc_label": patent["cpc_label"],
                "overlap": overlap,
            })

    return sorted(
        results,
        key=lambda x: x["overlap"],
        reverse=True
    )[:limit]

for item in TEST_QUERIES[:2]:
    print(f"\nQUERY: {item['query']}")

    for result in lexical_candidates(item["query"]):
        print(
            f"- {result['patent_number']} | "
            f"{result['title']} | "
            f"overlap={result['overlap']}"
        )


QUERY: Optical network terminals for passive optical networks
- 14907476 | Access Network, Selection and Connection Methods, Devices, and Computer Programs | overlap=6
- 15003824 | FLOOR COVERING SYSTEM WITH SENSORS | overlap=6
- 15004118 | ROBOTIC ASSISTANCE IN SECURITY MONITORING | overlap=6
- 15004740 | METHODS, SYSTEMS AND APPARATUS FOR ENABLING AN ACCESSORY FOR USE WITH A HOST DEVICE | overlap=6
- 15005348 | Method and Apparatus for Electron Beam Lithography | overlap=6

QUERY: Methods for treating rosacea using topical compositions
- 13144833 | ROSACEA TREATMENTS AND KITS FOR PERFORMING THEM | overlap=7
- 14907479 | FORMULATIONS FOR EPIDERMAL REPAIR | overlap=7
- 14907706 | METHOD FOR TREATING SKIN THICKENING | overlap=7
- 14907811 | Tetrapeptides Derived from Human C-X-C Chemokines Useful for Treatment of Various Skin Conditions | overlap=7
- 15004153 | ZINC OXIDE COMPLEXES | overlap=7


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

search_fields = ["title", "abstract", "claims", "description"]

validation_texts = []

for patent in validation:
    text = "\n\n".join(
        f"{field.upper()}:\n{patent.get(field) or ''}"
        for field in search_fields
    )
    validation_texts.append(text)

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=100_000,
    ngram_range=(1, 2),
    dtype=np.float32,
)

validation_matrix = tfidf.fit_transform(validation_texts)

print("Validation patents:", len(validation_texts))
print("TF-IDF vocabulary size:", len(tfidf.vocabulary_))
print("Matrix shape:", validation_matrix.shape)

Validation patents: 9094
TF-IDF vocabulary size: 100000
Matrix shape: (9094, 100000)
